# Does Selene carry the paper's noise model?

This notebook tests Selene's two noise parameters one at a time, on one or two physical
qubits, against Stim channels whose meaning is documented. Selene runs use the `Quest`
state-vector simulator with a `DepolarizingErrorModel` carrying only the parameter under test.

**Result.** Selene's `p_meas` behaves like the paper's `p_read`: a flip of the reported bit
only. Selene's `p_1q` is depolarizing, where the paper's data channel is a pure bit-flip (X), and
Selene has no parameter that gives a pure bit-flip.

The paper's model (SM A) applies two independent faults per round and simulates no ancilla:

| channel | acts on | probability |
| --- | --- | --- |
| data | X on each data qubit | `p_data = 1 − (1 − p)(1 − p_idle)`, `p_idle = 1 − exp(−p·λ·Δt)` (≈ `p(1 + λΔt)` for small `p`) |
| syndrome | the recorded bit only | `p_read = b_read · p` |

| | question | answer | evidence |
| --- | --- | --- | --- |
| **(a)** | can Selene apply bit-flip-only (X) data noise? | **no** | X-basis flip rate after 40 idle ops is 0.3476, 163σ above zero (section 2) |
| **(b)** | is `p_meas` an independent flip of the recorded bit? | **yes** | Bell-pair disagreement within 1.5σ of `2p(1−p)` at all 6 values of `p_meas` (section 3) |

## 0. Setup

In [3]:
import numpy as np
import pandas as pd
import stim

from guppylang import guppy
from guppylang.std.angles import angle
from guppylang.std.builtins import output, py
from guppylang.std.quantum import cx, h, measure, qubit
from guppylang.std.qsystem import phased_x
from selene_sim import DepolarizingErrorModel, Quest, build

SHOTS = 50_000
pd.set_option('display.width', 130)

_BUILT = {}


def selene(key, make, shots=SHOTS, seed=1, **channels):
    """Run a Guppy program under a model carrying only the named channels."""
    if key not in _BUILT:
        _BUILT[key] = build(make().compile())
    return [dict(s) for s in _BUILT[key].run_shots(
        simulator=Quest(), n_qubits=4, n_shots=shots,
        error_model=DepolarizingErrorModel(random_seed=seed, **channels),
        random_seed=seed)]


def sigma(p, n=SHOTS):
    return float(np.sqrt(max(p * (1 - p), 1e-12) / n))


def accumulate(n, f):
    """P(odd number of flips) after n independent flips of probability f."""
    return (1 - (1 - 2 * f) ** n) / 2


## 1. The same circuit, both stacks

Prepare `|0⟩`, apply `n` idle operations, measure in Z.

Selene's error model has no duration term: noise is charged per operation, so an idle unit
has to be an explicit gate. `phased_x(q, 0, 0)` is used because the compiler keeps it; `x;x`,
`h;h` and `z;z` are cancelled. Each one is charged `p_1q`.

The two Stim references:
- `DEPOLARIZE1(p)`: X, Y or Z, each with probability `p/3`. Z-basis flip rate per op: `2p/3`,
  because X and Y flip `|0⟩` and Z does not.
- `X_ERROR(p)`: X with probability `p`. Z-basis flip rate per op: `p`.

After `n` ops the flip rate is `accumulate(n, f) = (1 − (1 − 2f)^n) / 2`, for `f = 2p/3` or `f = p`.

In [4]:
def guppy_idle(n, basis):
    """Prepare, n idle ops, measure. The basis is chosen in Python: a Guppy program
    cannot branch on a string, and there is no reason for it to."""
    if basis == "Z":
        @guppy
        def prog() -> None:
            q = qubit()
            for _ in range(py(n)):
                phased_x(q, angle(0.0), angle(0.0))
            output("m", measure(q).read())
    else:
        @guppy
        def prog() -> None:
            q = qubit()
            h(q)
            for _ in range(py(n)):
                phased_x(q, angle(0.0), angle(0.0))
            h(q)
            output("m", measure(q).read())
    return prog


def stim_idle(n, p, channel, basis):
    c = stim.Circuit()
    c.append('R', [0])
    if basis == "X":
        c.append('H', [0])
    for _ in range(n):
        c.append(channel, [0], p)
    if basis == "X":
        c.append('H', [0])
    c.append('M', [0])
    return float(c.compile_sampler(seed=1).sample(SHOTS)[:, 0].mean())


P = 0.02
NS = (0, 1, 2, 5, 10, 20, 40)

rows = []
for n in NS:
    sel = selene(("idle", n, "Z"), lambda n=n: guppy_idle(n, "Z"), p_1q=P)
    rows.append({
        'n_ops': n,
        'selene p_1q': sum(s['m'] for s in sel) / SHOTS,
        'stim DEPOLARIZE1': stim_idle(n, P, 'DEPOLARIZE1', "Z"),
        'stim X_ERROR': stim_idle(n, P, 'X_ERROR', "Z"),
        'analytic 2p/3': accumulate(n, 2 * P / 3),
        'analytic p': accumulate(n, P),
    })

z = pd.DataFrame(rows).set_index('n_ops').round(4)
print(f"Z-basis flip rate, p = {P}, {SHOTS} shots (sigma ~ {sigma(0.1):.4f})\n")
z


Z-basis flip rate, p = 0.02, 50000 shots (sigma ~ 0.0013)



,selene p_1q,stim DEPOLARIZE1,stim X_ERROR,analytic 2p/3,analytic p
n_ops,,,,,
0,0.0000,0.0000,0.0000,0.0000,0.0000
1,0.0136,0.0132,0.0207,0.0133,0.0200
2,0.0269,0.0260,0.0394,0.0263,0.0392
5,0.0637,0.0629,0.0914,0.0632,0.0923
10,0.1194,0.1178,0.1682,0.1184,0.1676
20,0.2103,0.2086,0.2812,0.2088,0.2790
40,0.3324,0.3272,0.4050,0.3304,0.4023


Selene's `p_1q` matches `DEPOLARIZE1` and `accumulate(n, 2p/3)` to within 2σ at every `n`
(largest deviation −1.9σ at `n = 5`). It does not match `X_ERROR`: it sits 9.5σ below it at
`n = 1` and 24.6σ below it at `n = 40`. So `p_1q = p` in Selene is the same channel as
`DEPOLARIZE1(p)` in Stim, checked over flip rates from 0.012 (`n = 1`) to 0.33 (`n = 40`).

## 2. (a) Bit-flip, or depolarizing?

The Z basis cannot tell the two channels apart: both flip `|0⟩`, only at different rates. The
X basis can, because a pure bit-flip leaves `|+⟩` unchanged:

| channel | Z-basis flip rate per op | X-basis flip rate per op |
| --- | --- | --- |
| depolarizing `p` | `2p/3` | `2p/3` |
| bit-flip `p` | `p` | **0** |

So any non-zero X-basis rate means the channel contains Z errors.

**Expect a floor at `n = 0`.** The X-basis circuit needs two `h` gates, and Selene charges each
native operation it compiles `h` into. Stim's `H` is noiseless, so Stim has no floor. Rather
than subtract the floor, fit it: if each `h` compiles to `k` charged operations, every point
should lie on `accumulate(n + 2k, 2p/3)`.

In [ ]:
rows = []
for n in NS:
    sel = selene(("idle", n, "X"), lambda n=n: guppy_idle(n, "X"), p_1q=P)
    rows.append({
        'n_ops': n,
        'selene p_1q': sum(s['m'] for s in sel) / SHOTS,
        'stim DEPOLARIZE1': stim_idle(n, P, 'DEPOLARIZE1', "X"),
        'stim X_ERROR': stim_idle(n, P, 'X_ERROR', "X"),
        'analytic, n ops': accumulate(n, 2 * P / 3),
        'analytic, n+4 ops': accumulate(n + 4, 2 * P / 3),
    })

x = pd.DataFrame(rows).set_index('n_ops').round(4)
print(f"X-basis flip rate, p = {P}, {SHOTS} shots\n")
print(x)

resid = (x['selene p_1q'] - x['analytic, n+4 ops']).abs().max()
print(f"\nlargest deviation from the n+4 curve: {resid:.4f}  (sigma ~ {sigma(0.2):.4f})")
print("so two h gates cost 4 charged operations: each h is 2 native ops, not 1.\n")

x40 = x.loc[40, 'selene p_1q']
print(f"Selene at n=40 in the X basis: {x40:.4f}  ({x40 / sigma(x40):.0f} sigma from zero)")
print("stim X_ERROR in the X basis:", x.loc[40, 'stim X_ERROR'],
      "- exactly zero, as a bit-flip channel must be")
print("\n(a) VERDICT: Selene's only data channel is depolarizing. Bit-flip is unreachable.")


X-basis flip rate, p = 0.02, 50000 shots

       selene p_1q  stim DEPOLARIZE1  stim X_ERROR  analytic, n ops  analytic, n+4 ops
n_ops                                                                                 
0           0.0497            0.0000           0.0           0.0000             0.0512
1           0.0628            0.0139           0.0           0.0133             0.0632
2           0.0742            0.0275           0.0           0.0263             0.0749
5           0.1071            0.0645           0.0           0.0632             0.1080
10          0.1582            0.1191           0.0           0.1184             0.1575
20          0.2380            0.2082           0.0           0.2088             0.2386
40          0.3476            0.3317           0.0           0.3304             0.3478

largest deviation from the n+4 curve: 0.0015  (sigma ~ 0.0018)
so two h gates cost 4 charged operations: each h is 2 native ops, not 1.

Selene at n=40 in the X basis: 0.3476

**(a) No.** At `n = 40` the X-basis flip rate is 0.3476, 163σ above zero and equal to the
Z-basis rate (0.3318) plus the contribution of the four `h` operations. That is a symmetric
channel. `X_ERROR` gives exactly 0.0 in the X basis. Selene's `DepolarizingErrorModel` has no
parameter that biases it; a bit-flip-only data channel would need a compiled error-model plugin.

What this means for a memory that decodes only the Z-check sector: X and Y both flip a Z-check
and Z does not, so depolarizing at `p_1q` acts on that sector as a bit-flip at `2·p_1q/3`. The Z
errors are still applied, but no Z-check detects them. The substitution is valid for a single
sector (Z-basis memory, Z-checks decoded); it is wrong for any experiment that decodes both.

**Each `h` compiles to 2 charged operations.** With `k = 2`, all seven points lie on
`accumulate(n + 4, 2p/3)` within 1.6σ (largest deviation 0.0015 against σ ≈ 0.0018). `k = 1`
misses by up to 33σ and `k = 3` by up to 21σ. So a gate count in the Guppy source is not the
number of charged operations; the charged count has to be measured. Notebook 04 measures it for
a full syndrome-extraction round.

## 3. (b) What is `p_meas`?

Two readings are possible: a flip of the recorded bit (the paper's `p_read`, Stim's `M(p)`), or
an X applied to the qubit just before it is measured.

**These two cannot be told apart in Guppy.** Measurement is destructive: `measure` consumes the
qubit and `measure_and_reset` overwrites it. X-then-measure and measure-then-flip-the-record
differ only in the post-measurement state of a qubit that no longer exists.

The question that does change results is whether the flips are **independent, one per
measurement**, leaving the pre-measurement state undisturbed. A Bell pair tests it: its two
outcomes are perfectly correlated, so independent flips at `p` make them disagree with
probability `2p(1 − p)`.

In [4]:
@guppy
def prep_measure() -> None:
    q = qubit()
    output("m", measure(q).read())


@guppy
def bell() -> None:
    a, b = qubit(), qubit()
    h(a)
    cx(a, b)
    output("ma", measure(a).read())
    output("mb", measure(b).read())


def stim_bell(p):
    c = stim.Circuit()
    c.append('R', [0, 1])
    c.append('H', [0])
    c.append('CX', [0, 1])
    c.append('M', [0, 1], p)
    s = c.compile_sampler(seed=1).sample(SHOTS)
    return float((s[:, 0] != s[:, 1]).mean())


rows = []
for pm in (0.0, 0.02, 0.05, 0.1, 0.2, 0.4):
    single = selene(("prep",), lambda: prep_measure, p_meas=pm)
    pair = selene(("bell",), lambda: bell, p_meas=pm)
    rows.append({
        'p_meas': pm,
        'selene P(read 1)': sum(s['m'] for s in single) / SHOTS,
        'selene P(disagree)': sum(1 for s in pair if s['ma'] != s['mb']) / SHOTS,
        'stim M(p) P(disagree)': stim_bell(pm),
        'analytic 2p(1-p)': 2 * pm * (1 - pm),
    })

m = pd.DataFrame(rows).set_index('p_meas').round(4)
print(f"{SHOTS} shots\n")
print(m)
print("\n(b) VERDICT: p_meas is an independent per-measurement flip of the reported bit.")
print("    It matches stim M(p) and the paper's p_read.")


50000 shots

        selene P(read 1)  selene P(disagree)  stim M(p) P(disagree)  analytic 2p(1-p)
p_meas                                                                               
0.00              0.0000              0.0000                 0.0000            0.0000
0.02              0.0196              0.0389                 0.0399            0.0392
0.05              0.0507              0.0931                 0.0949            0.0950
0.10              0.1005              0.1799                 0.1794            0.1800
0.20              0.2003              0.3202                 0.3218            0.3200
0.40              0.3990              0.4827                 0.4811            0.4800

(b) VERDICT: p_meas is an independent per-measurement flip of the reported bit.
    It matches stim M(p) and the paper's p_read.


**(b) Yes.** Across the six values of `p_meas` from 0 to 0.4:
- single-qubit `P(read 1)` equals `p_meas` within 0.8σ;
- Bell-pair disagreement equals `2p(1 − p)` within 1.5σ;
- Stim's `M(p)` agrees with the same curve within 0.9σ.

So `p_meas` is an independent flip of the recorded bit, one per measurement.

`p_meas` does *not* model a fault that spreads to other qubits. In Stim that channel is
`after_reset_flip_probability`: a real X on an ancilla after reset, which then propagates
through the extraction CNOTs onto the data. Selene has no equivalent parameter, and the paper
does not include one either.

## 4. Summary

| paper | Stim | Selene | match |
| --- | --- | --- | --- |
| data fault, X with `p_data` | `X_ERROR(p_data)` | `p_1q`, depolarizing | Z-check sector only, with `p_1q = 3·p_data/2` |
| syndrome flip `p_read` | `M(p_read)` | `p_meas` | exact, within 1.5σ at 6 values |
| fault on an ancilla after reset | `after_reset_flip_probability` | not available | absent from the paper too |

1. **Selene's `p_1q` is standard depolarizing.** It matches `DEPOLARIZE1` within 2σ for flip
   rates from 0.012 to 0.33, so a `p_1q` value means the same thing in both simulators.
2. **Selene cannot produce bit-flip-only data noise.** The X-basis rate is 163σ from zero at
   `n = 40`, and `DepolarizingErrorModel` has no bias parameter.
3. **Selene's `p_meas` is the paper's `p_read`.**
4. **The only mismatch is the data channel**, and it is removed by the substitution
   `p_1q = 3·p_data/2`, valid only while a single sector is decoded.
5. **Charged operations must be measured, not counted from source.** One `h` is 2 charged
   operations.

### Consequence

The paper's model is bit-flip only, so only Stim can reproduce it exactly, with `X_ERROR` and
`M(p)`. Selene can reproduce it only through the single-sector substitution above. Selene is
still needed for what Stim cannot express: the extraction circuit compiled to native gates,
measurement-conditioned control flow, and leakage heralds. Throughput was not measured here.

### Follow-ups

1. **`p_stab` in the Stim τ sweep: done** in `05_surface_code_interval.ipynb`, which restores
   `p_data = 1 − (1 − p_stab)(1 − p_idle)` with `p_stab = p`.
2. **The `2p/3` substitution on an encoded block: not tested here.** Notebooks 06 and 07 rely
   on it: their decoder priors use `f = 2·p_1q/3` per charged operation
   (`surface_memory.Geometry.priors`).